In [61]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

In [62]:
df = pd.read_csv("C:/Users/Eduardo Filho/Desktop/projeto ICD/Trabalho-ICD-/CSVs/Dados unidos/dados_mensais_unificado.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17382 entries, 0 to 17381
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Categoria                17382 non-null  str  
 1   Especificação            17301 non-null  str  
 2   Porcentagens             17382 non-null  str  
 3   Incrementos/Decrementos  17382 non-null  str  
 4   Mês                      17382 non-null  str  
 5   Ano                      17382 non-null  int64
dtypes: int64(1), str(5)
memory usage: 814.9 KB


In [63]:
df.head(5)

,Categoria,Especificação,Porcentagens,Incrementos/Decrementos,Mês,Ano
0,OS Version,Windows,95.92%,+0.06%,Janeiro,2019
1,OS Version,Windows 10 64 bit,63.77%,-0.02%,Janeiro,2019
2,OS Version,Windows 7 64 bit,26.40%,+0.32%,Janeiro,2019
3,OS Version,Windows 8.1 64 bit,3.43%,-0.10%,Janeiro,2019
4,OS Version,Windows 7,1.52%,-0.09%,Janeiro,2019


### Valores nulos da coluna Espacificação.

Os valores nulos da coluna especificação possui a característica de ser todos pertencentes a categoria Video Card Description, alguns desses valores possuem uma porcentagem significativa outros nem tanto. Porém, ao procurar a fonte dos dados é possível identificar que é um erro da própria base de dados. Há duas opções: fazer a procura desses dados manualmente para substituir ou apagar as linnhas.

In [64]:
print(df[df["Especificação"].isnull()])

                    Categoria Especificação Porcentagens  \
140    Video Card Description           NaN        0.24%   
334    Video Card Description           NaN        0.23%   
523    Video Card Description           NaN        0.25%   
713    Video Card Description           NaN        0.25%   
902    Video Card Description           NaN        0.27%   
...                       ...           ...          ...   
15977  Video Card Description           NaN        1.77%   
16193  Video Card Description           NaN        1.93%   
16408  Video Card Description           NaN        1.78%   
16628  Video Card Description           NaN        1.05%   
16803  Video Card Description           NaN        2.13%   

      Incrementos/Decrementos        Mês   Ano  
140                    +0.01%    Janeiro  2019  
334                    -0.01%  Fevereiro  2019  
523                    +0.02%      Março  2019  
713                     0.00%      Abril  2019  
902                    +0.02%     

In [65]:
df["Porcentagens"] = df["Porcentagens"].str.replace("%", "")
df["Porcentagens"] = df["Porcentagens"].astype(float)



In [66]:
df["Incrementos/Decrementos"] = df["Incrementos/Decrementos"].str.replace("%","")
df["Incrementos/Decrementos"] = df["Incrementos/Decrementos"].str.replace("+","")
df["Incrementos/Decrementos"] = df["Incrementos/Decrementos"].astype(float)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17382 entries, 0 to 17381
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Categoria                17382 non-null  str    
 1   Especificação            17301 non-null  str    
 2   Porcentagens             17382 non-null  float64
 3   Incrementos/Decrementos  17382 non-null  float64
 4   Mês                      17382 non-null  str    
 5   Ano                      17382 non-null  int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 814.9 KB


In [67]:
a = df[df["Especificação"].isnull()]
print(f'Linhas faltantes: {len(a)}')
for i in range(len(a)):
    print(f" Categoria: {a["Categoria"].iloc[i]} Porcentagem: {a['Porcentagens'].iloc[i]}, mês: {a['Mês'].iloc[i]}, ano: {a['Ano'].iloc[i]}")

Linhas faltantes: 81
 Categoria: Video Card Description Porcentagem: 0.24, mês: Janeiro, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.23, mês: Fevereiro, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.25, mês: Março, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.25, mês: Abril, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.27, mês: Maio, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.25, mês: Junho, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.26, mês: Julho, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.31, mês: Agosto, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.34, mês: Setembro, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.24, mês: Outubro, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.2, mês: Novembro, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.16, mês: Dezembro, ano: 2019
 Categoria: Video Card Description Porcentagem: 0.2

### Ideia para resolução dos valores nulos. 

Ao analisar os dados: a porcentagem final de usuários de uma determinada categoria depende do seu valor no mês anterior e o seu incremento para aquele mês. A ideia é fazer o caminho inverso para tentar encontrar esse componente: Encontrar um valor que subtraído o incremento daquele mês possa encontrar um compomente do mês anterior e substituir o valor nulo por ele. 

In [68]:
disc = {"Janeiro" : 0,
       "Fevereiro" : 1,
       "Março" : 2,
       "Abril" : 3,
       "Maio" : 4,
       "Junho" : 5,
       "Julho" : 6,
       "Agosto" : 7,
       "Setembro" : 8,
       "Outubro" : 9,
       "Novembro" : 10,
       "Dezembro" : 11,
       }

### Substituição dos valores com base no Mês anterior.

Esse código possui alguns desafios: O mesmo hardware não possui nome em meses consecutivos, então muitas vezes um mês anterior não resolve. Além disso em alguns casos vários hardwares seriam possíveis para substituir o mesmo dado faltante com base nos cálculos ou não podiam ser preenchido por causa da falta de dados de 2018 para trás.

In [69]:
mascara_video = df["Categoria"] == "Video Card Description"
total_nulos_inicial = df.loc[mascara_video, "Especificação"].isnull().sum()

for i in range(20):
    video = df[mascara_video].copy()
    video["chave"] = video["Ano"] * 12 + video["Mês"].map(disc)

    nulos = video[video["Especificação"].isnull()]
    if nulos.empty:
        break

    preenchidos = 0
    for idx, linha in nulos.iterrows():
        valor_alvo = linha["Porcentagens"] - linha["Incrementos/Decrementos"]
        chave_anterior = linha["chave"] - 1

        candidatos = video[
            (video["chave"] == chave_anterior)
            & np.isclose(video["Porcentagens"], valor_alvo, atol=0.005)
            & video["Especificação"].notnull()
        ]
        especificacoes = candidatos["Especificação"].unique()

        if len(especificacoes) == 1:
            df.loc[idx, "Especificação"] = especificacoes[0]
            preenchidos += 1

    if preenchidos == 0:
        break

total_nulos_final = df.loc[mascara_video, "Especificação"].isnull().sum()
print(f"Nulos preenchidos: {total_nulos_inicial - total_nulos_final} de {total_nulos_inicial}")

Nulos preenchidos: 35 de 81


In [70]:
restantes = df[mascara_video & df["Especificação"].isnull()]
print(f"Linhas ainda nulas: {len(restantes)}")
restantes[["Categoria", "Porcentagens", "Incrementos/Decrementos", "Mês", "Ano"]]

Linhas ainda nulas: 46


,Categoria,Porcentagens,Incrementos/Decrementos,Mês,Ano
140,Video Card Description,0.24,0.01,Janeiro,2019
334,Video Card Description,0.23,-0.01,Fevereiro,2019
713,Video Card Description,0.25,0.00,Abril,2019
902,Video Card Description,0.27,0.02,Maio,2019
1092,Video Card Description,0.25,-0.02,Junho,2019
1851,Video Card Description,0.24,0.01,Outubro,2019
2054,Video Card Description,0.20,-0.04,Novembro,2019
2247,Video Card Description,0.16,-0.04,Dezembro,2019
2426,Video Card Description,0.20,0.01,Janeiro,2020
2629,Video Card Description,0.19,-0.01,Fevereiro,2020


### Remoção das linhas nulas restantes

In [72]:
df = df.drop(restantes.index).reset_index(drop=True)
print(f"Linhas removidas: {len(restantes)}")
print(f"Nulos restantes em Especificação: {df['Especificação'].isnull().sum()}")

Linhas removidas: 46
Nulos restantes em Especificação: 0


In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17336 entries, 0 to 17335
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Categoria                17336 non-null  str    
 1   Especificação            17336 non-null  str    
 2   Porcentagens             17336 non-null  float64
 3   Incrementos/Decrementos  17336 non-null  float64
 4   Mês                      17336 non-null  str    
 5   Ano                      17336 non-null  int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 812.8 KB
